# Reacher DPF Training and Inference Workflow

This self-contained notebook records the workflow for training and running inference with the exploration-trained DPF on the Reacher task, including optional structured-HNN guidance and target-based guidance.

Shared dataset copy:

`/Data/gsang/hnn_guided_dpf/reacher_dpf/reacher_exploration_iid_uniform_traj40000_val4000_len1000_v1`

The notebook contains the Reacher-only DPF dataset loader, concat-state Perceiver architecture, structured HNN, training loops, normalization utilities, DDIM prefix-completion sampling, HNN-based guidance, and L1/L2/Linf target-based guidance.


## 1. Setup

This cell is portable across users on the same server. The only required absolute paths are the shared dataset and optional pretrained checkpoint paths on `/Data`; outputs go to the current user's home directory.


In [ ]:
from pathlib import Path
import json
import sys

# Shared dataset path on this server. The notebook only needs read access here.
DATA_DIR = Path('/Data/gsang/hnn_guided_dpf/reacher_dpf/reacher_exploration_iid_uniform_traj40000_val4000_len1000_v1')
TRAIN_H5 = DATA_DIR / 'train_traj_40000-steps_1000.h5'
VAL_H5 = DATA_DIR / 'val_traj_4000-steps_1000.h5'
BUILD_REPORT = DATA_DIR / 'build_report.json'
# These shared checkpoints make inference runnable immediately, even before retraining.
SHARED_PRETRAINED_CKPT = Path('/Data/gsang/hnn_guided_dpf/reacher_dpf/checkpoints/dpf_exploration_iid_uniform_len1000_v1/trajectory_dpf_x0Stabilized&AbsoluteTimeEncoding&VariableTrajLength&UniformContext&EncoderNone&DecoderAttentions_cond-concat_torque_in_state_layout-shifted_tau_qpos-raw:epoch=2999_val_loss:val_loss=0.0983.ckpt')

SHARED_HNN_CKPT = Path('/Data/gsang/hnn_guided_dpf/reacher_dpf/checkpoints/hnn_exploration_iid_uniform_len1000_v1/StructuredHNN-ReacherExploration-IID-epoch-epoch=999.ckpt')

# User-local outputs. This avoids writing checkpoints into another user's home directory.
RUN_ROOT = Path.home() / 'hnn_guided_dpf_reacher_dpf_runs'
CKPT_DIR = RUN_ROOT / 'checkpoints/reacher/dpf_exploration_iid_uniform_len1000_v1'
HNN_CKPT_DIR = RUN_ROOT / 'checkpoints/reacher/hnn_exploration_iid_uniform_len1000_v1'
EVAL_OUTPUT_DIR = RUN_ROOT / 'plots/reacher_dpf_unguided_notebook_eval'
CKPT_DIR.mkdir(parents=True, exist_ok=True)
HNN_CKPT_DIR.mkdir(parents=True, exist_ok=True)
EVAL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Used for checkpoint loading, sampling, and optional training. Change if GPU 0 is busy.
DEVICE = 'cuda:0'  # change to 'cpu' or another CUDA device if needed

print('python:', sys.executable)
print('data dir:', DATA_DIR)
print('train h5 exists:', TRAIN_H5.exists(), TRAIN_H5)
print('val h5 exists:', VAL_H5.exists(), VAL_H5)
print('shared DPF checkpoint exists:', SHARED_PRETRAINED_CKPT.exists(), SHARED_PRETRAINED_CKPT)
print('shared HNN checkpoint exists:', SHARED_HNN_CKPT.exists(), SHARED_HNN_CKPT)
print('run root:', RUN_ROOT)


## 2. Dataset Sanity Check

The DPF was trained on IID-uniform torque exploration trajectories. Each trajectory stores `qpos`, `mom`, `qvel`, `torque`, and fingertip position sequences.


In [ ]:
import h5py

with h5py.File(TRAIN_H5, 'r') as f:
    print('File attrs:')
    for key in ['num_trajectories', 'num_steps', 'dt', 'data_dt', 'generator_config']:
        print(f'  {key}:', f.attrs.get(key))

    first_key = sorted(k for k in f.keys() if k.startswith('traj_'))[0]
    print('\nFirst trajectory group:', first_key)
    for name, ds in f[first_key].items():
        print(f'  {name:18s} shape={tuple(ds.shape)} dtype={ds.dtype}')

## 3. Self-Contained Reacher DPF Implementation

This cell embeds only the code needed for the shared Reacher DPF setup: raw qpos, cached HDF5 trajectories, concat-state shifted-torque Perceiver DPF training, checkpoint loading, and DDIM prefix-completion inference.


In [ ]:
# Reacher-only, self-contained DPF implementation.
# This is intentionally narrow: raw-qpos Reacher, concat-state shifted-tau DPF,
# training, checkpoint loading, and prefix-completion inference.

import math
from pathlib import Path
from typing import Optional, Tuple

import h5py
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import pytorch_lightning as pl
from pytorch_lightning.callbacks import ModelCheckpoint
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
from perceiver.model.core import FourierPositionEncoding

DEFAULT_NORMALIZATION_RANGE_EPSILON = 1e-2
RAW_QPOS = 'raw'


def infer_qpos_representation_from_xml(xml_content: Optional[str]) -> str:
    return RAW_QPOS


# The shared checkpoint was trained with raw joint angles, not sin/cos encoded qpos.
def encode_qpos_array(qpos: np.ndarray, qpos_representation: str) -> np.ndarray:
    if qpos_representation != RAW_QPOS:
        raise ValueError('This notebook only supports raw Reacher qpos.')
    return np.asarray(qpos, dtype=np.float32)


def decode_qpos_tensor(qpos: torch.Tensor, qpos_representation: str) -> torch.Tensor:
    if qpos_representation != RAW_QPOS:
        raise ValueError('This notebook only supports raw Reacher qpos.')
    return qpos


def raw_qpos_dim(encoded_qpos_dim_value: int, qpos_representation: str) -> int:
    if qpos_representation != RAW_QPOS:
        raise ValueError('This notebook only supports raw Reacher qpos.')
    return int(encoded_qpos_dim_value)


def override_qpos_normalization_stats(qpos_min, qpos_max, qpos_representation: str):
    if qpos_representation != RAW_QPOS:
        raise ValueError('This notebook only supports raw Reacher qpos.')
    return qpos_min, qpos_max


class TrajectoryDPFCached(Dataset):
    """In-memory HDF5 dataset used by the Reacher DPF training workflow."""

    def __init__(self, h5_path: str, trajectory_length: int = 1000, qpos_representation_override: Optional[str] = None):
        super().__init__()
        with h5py.File(h5_path, 'r') as f:
            self.num_traj = int(f.attrs['num_trajectories'])
            self.num_steps = int(f.attrs['num_steps'])
            self.dt = float(f.attrs.get('dt', 0.0001))
            self.data_dt = float(f.attrs.get('data_dt', 0.0002))
            self.xml = f.attrs.get('xml', None)
            self.qpos_representation = qpos_representation_override or infer_qpos_representation_from_xml(self.xml)
            if self.qpos_representation != RAW_QPOS:
                raise ValueError('Use qpos_representation_override="raw" for this Reacher package.')

            # Cache complete trajectories in memory so training batches avoid repeated H5 reads.
            qpos, mom, torque = [], [], []
            for i in range(self.num_traj):
                traj = f[f'traj_{i}']
                qpos.append(encode_qpos_array(traj['seq_qpos'][:trajectory_length], RAW_QPOS))
                mom.append(traj['seq_mom'][:trajectory_length])
                torque.append(traj['seq_torque'][:trajectory_length])

        self.all_seq_qpos = torch.from_numpy(np.asarray(qpos, dtype=np.float32))
        self.all_seq_mom = torch.from_numpy(np.asarray(mom, dtype=np.float32))
        self.all_seq_torque = torch.from_numpy(np.asarray(torque, dtype=np.float32))
        print('---------------Statistics--------------')
        print(f'Range of qpos: [{self.all_seq_qpos.max()} - {self.all_seq_qpos.min()}]; Std of qpos: {self.all_seq_qpos.std()}')
        print(f'Range of mom: [{self.all_seq_mom.max()} - {self.all_seq_mom.min()}]; Std of mom: {self.all_seq_mom.std()}')
        print(f'Range of torque: [{self.all_seq_torque.max()} - {self.all_seq_torque.min()}]; Std of torque: {self.all_seq_torque.std()}')

    def __len__(self):
        return self.num_traj

    def __getitem__(self, index):
        return {
            'seq_qpos': self.all_seq_qpos[index],
            'seq_mom': self.all_seq_mom[index],
            'seq_torque': self.all_seq_torque[index],
        }


def compute_normalization_stats(dataloader, qpos_dim, mom_dim, torque_dim, max_timesteps, qpos_representation: str = RAW_QPOS):
    """Compute per-dimension min/max normalization stats for qpos, momentum, and torque."""
    qpos_min = torch.full((qpos_dim,), float('inf'))
    qpos_max = torch.full((qpos_dim,), float('-inf'))
    mom_min = torch.full((mom_dim,), float('inf'))
    mom_max = torch.full((mom_dim,), float('-inf'))
    torque_min = torch.full((torque_dim,), float('inf'))
    torque_max = torch.full((torque_dim,), float('-inf'))
    count = 0

    # DPF predicts normalized state noise, so these ranges must match the training data.
    for batch in tqdm(dataloader, desc='Computing stats'):
        qpos = batch['seq_qpos']
        mom = batch['seq_mom']
        torque = batch['seq_torque']
        qpos_min = torch.min(qpos_min, qpos.amin(dim=(0, 1)))
        qpos_max = torch.max(qpos_max, qpos.amax(dim=(0, 1)))
        mom_min = torch.min(mom_min, mom.amin(dim=(0, 1)))
        mom_max = torch.max(mom_max, mom.amax(dim=(0, 1)))
        torque_min = torch.min(torque_min, torque.amin(dim=(0, 1)))
        torque_max = torch.max(torque_max, torque.amax(dim=(0, 1)))
        count += int(qpos.shape[0])

    qpos_min, qpos_max = override_qpos_normalization_stats(qpos_min, qpos_max, qpos_representation)
    eps = DEFAULT_NORMALIZATION_RANGE_EPSILON
    qpos_max = torch.where((qpos_max - qpos_min) < eps, qpos_min + eps, qpos_max)
    mom_max = torch.where((mom_max - mom_min) < eps, mom_min + eps, mom_max)
    torque_max = torch.where((torque_max - torque_min) < eps, torque_min + eps, torque_max)
    print(f'Computed stats for {count} trajectories, {max_timesteps} timesteps each')
    return qpos_min, qpos_max, mom_min, mom_max, torque_min, torque_max


class EMA:
    """Small exponential-moving-average helper for optional sampling with EMA weights."""

    def __init__(self, model: nn.Module, decay: float = 0.9995):
        self.decay = decay
        self.model = model
        self.shadow = {name: p.data.clone().detach() for name, p in model.named_parameters() if p.requires_grad}
        self.backup = {}

    def update(self, model=None):
        for name, p in self.model.named_parameters():
            if not p.requires_grad:
                continue
            old = self.shadow.get(name, p.data.clone().detach()).to(device=p.device, dtype=p.dtype)
            self.shadow[name] = ((1.0 - self.decay) * p.data + self.decay * old).clone().detach()

    def store(self, model=None):
        self.backup = {}
        for name, p in self.model.named_parameters():
            if p.requires_grad and name in self.shadow:
                self.backup[name] = p.data.clone().detach()

    def copy_to(self, model=None):
        for name, p in self.model.named_parameters():
            if p.requires_grad and name in self.shadow:
                p.data = self.shadow[name].to(device=p.device, dtype=p.dtype).clone().detach()

    def restore(self, model=None):
        for name, p in self.model.named_parameters():
            if p.requires_grad and name in self.backup:
                p.data = self.backup[name].clone().detach()
        self.backup = {}


# These AdaLN attention blocks are included because the shared checkpoint was trained with them.
class AdaLNSelfAttentionBlock(nn.Module):
    def __init__(self, dim: int, cond_dim: int, num_heads: int = 8, dropout: float = 0.0):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = dim // num_heads
        self.norm1 = nn.LayerNorm(dim, elementwise_affine=False)
        self.norm2 = nn.LayerNorm(dim, elementwise_affine=False)
        self.qkv = nn.Linear(dim, dim * 3)
        self.proj = nn.Linear(dim, dim)
        self.attn_dropout = nn.Dropout(dropout)
        self.mlp = nn.Sequential(nn.Linear(dim, dim * 4), nn.GELU(), nn.Dropout(dropout), nn.Linear(dim * 4, dim))
        self.adaLN_modulation = nn.Sequential(nn.SiLU(), nn.Linear(cond_dim, dim * 6))
        nn.init.zeros_(self.adaLN_modulation[-1].weight)
        nn.init.zeros_(self.adaLN_modulation[-1].bias)

    def forward(self, x: torch.Tensor, cond: torch.Tensor) -> torch.Tensor:
        B, N, C = x.shape
        mod = self.adaLN_modulation(cond)
        scale1, shift1, gate1, scale2, shift2, gate2 = mod.chunk(6, dim=-1)
        h = self.norm1(x) * (1 + scale1) + shift1
        qkv = self.qkv(h).reshape(B, N, 3, self.num_heads, self.head_dim).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]
        attn = (q @ k.transpose(-2, -1)) * (self.head_dim ** -0.5)
        attn = self.attn_dropout(attn.softmax(dim=-1))
        attn = (attn @ v).transpose(1, 2).reshape(B, N, C)
        x = x + gate1 * self.proj(attn)
        h = self.norm2(x) * (1 + scale2) + shift2
        return x + gate2 * self.mlp(h)


class AdaLNCrossAttentionBlock(nn.Module):
    def __init__(self, query_dim: int, latent_dim: int, cond_dim: int, state_dim: int, num_heads: int = 8, dropout: float = 0.0):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = query_dim // num_heads
        self.state_dim = state_dim
        self.norm_q = nn.LayerNorm(query_dim, elementwise_affine=False)
        self.norm_kv = nn.LayerNorm(latent_dim)
        self.q_proj = nn.Linear(query_dim, query_dim)
        self.kv_proj = nn.Linear(latent_dim, query_dim * 2)
        self.out_proj = nn.Linear(query_dim, query_dim)
        self.attn_dropout = nn.Dropout(dropout)
        self.adaLN_modulation = nn.Sequential(nn.SiLU(), nn.Linear(cond_dim, state_dim * 2))
        nn.init.zeros_(self.adaLN_modulation[-1].weight)
        nn.init.zeros_(self.adaLN_modulation[-1].bias)

    def forward(self, queries: torch.Tensor, latents: torch.Tensor, cond: torch.Tensor) -> torch.Tensor:
        B, T, Cq = queries.shape
        N = latents.shape[1]
        scale, shift = self.adaLN_modulation(cond).chunk(2, dim=-1)
        queries_normed = self.norm_q(queries)
        state_normed = queries_normed[:, :, :self.state_dim]
        enc_normed = queries_normed[:, :, self.state_dim:]
        queries_modulated = torch.cat([state_normed * (1 + scale) + shift, enc_normed], dim=-1)
        q = self.q_proj(queries_modulated)
        kv = self.kv_proj(self.norm_kv(latents))
        k, v = kv.chunk(2, dim=-1)
        q = q.reshape(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        k = k.reshape(B, N, self.num_heads, self.head_dim).transpose(1, 2)
        v = v.reshape(B, N, self.num_heads, self.head_dim).transpose(1, 2)
        attn = (q @ k.transpose(-2, -1)) * (self.head_dim ** -0.5)
        attn = self.attn_dropout(attn.softmax(dim=-1))
        attn = (attn @ v).transpose(1, 2).reshape(B, T, Cq)
        return queries + self.out_proj(attn)


class ConditionedPerceiverEncoder(nn.Module):
    def __init__(self, num_input_channels: int, num_latents: int = 256, num_latent_channels: int = 256,
                 cond_dim: int = 256, state_dim: int = 6, num_self_attention_blocks: int = 8,
                 num_heads: int = 8, dropout: float = 0.0):
        super().__init__()
        self.num_latents = num_latents
        self.num_latent_channels = num_latent_channels
        self.latents = nn.Parameter(torch.randn(1, num_latents, num_latent_channels) * 0.02)
        self.cross_attn_norm_latent = nn.LayerNorm(num_latent_channels)
        self.cross_attn_norm_context = nn.LayerNorm(num_input_channels)
        self.cross_attn_q = nn.Linear(num_latent_channels, num_latent_channels)
        self.cross_attn_kv = nn.Linear(num_input_channels, num_latent_channels * 2)
        self.cross_attn_proj = nn.Linear(num_latent_channels, num_latent_channels)
        self.cross_attn_num_heads = num_heads
        self.cross_attn_head_dim = num_latent_channels // num_heads
        self.self_attn_blocks = nn.ModuleList([
            AdaLNSelfAttentionBlock(num_latent_channels, cond_dim, num_heads, dropout)
            for _ in range(num_self_attention_blocks)
        ])

    def forward(self, contexts: torch.Tensor, global_cond: torch.Tensor) -> torch.Tensor:
        B, N_ctx, _ = contexts.shape
        latents = self.latents.expand(B, -1, -1)
        q = self.cross_attn_q(self.cross_attn_norm_latent(latents))
        kv = self.cross_attn_kv(self.cross_attn_norm_context(contexts))
        k, v = kv.chunk(2, dim=-1)
        q = q.reshape(B, self.num_latents, self.cross_attn_num_heads, self.cross_attn_head_dim).transpose(1, 2)
        k = k.reshape(B, N_ctx, self.cross_attn_num_heads, self.cross_attn_head_dim).transpose(1, 2)
        v = v.reshape(B, N_ctx, self.cross_attn_num_heads, self.cross_attn_head_dim).transpose(1, 2)
        attn = (q @ k.transpose(-2, -1)) * (self.cross_attn_head_dim ** -0.5)
        attn = (attn.softmax(dim=-1) @ v).transpose(1, 2).reshape(B, self.num_latents, self.num_latent_channels)
        latents = latents + self.cross_attn_proj(attn)
        for block in self.self_attn_blocks:
            latents = block(latents, global_cond)
        return latents


class ConditionedPerceiverDecoder(nn.Module):
    def __init__(self, num_query_channels: int, num_latent_channels: int, num_output_channels: int,
                 state_dim: int, cond_dim: int = 256, num_heads: int = 8, dropout: float = 0.0,
                 num_decoder_blocks: int = 4):
        super().__init__()
        self.cross_attn = AdaLNCrossAttentionBlock(num_query_channels, num_latent_channels, cond_dim, state_dim, num_heads, dropout)
        self.self_attn_blocks = nn.ModuleList([
            AdaLNSelfAttentionBlock(num_query_channels, cond_dim, num_heads, dropout)
            for _ in range(num_decoder_blocks)
        ])
        self.output_norm = nn.LayerNorm(num_query_channels)
        self.output_proj = nn.Linear(num_query_channels, num_output_channels)

    def forward(self, latents: torch.Tensor, queries: torch.Tensor, cond: torch.Tensor) -> torch.Tensor:
        output = self.cross_attn(queries, latents, cond)
        for block in self.self_attn_blocks:
            output = block(output, cond)
        return self.output_proj(self.output_norm(output))


# The Perceiver sees trajectory tokens: [qpos, momentum, shifted torque, diffusion time, absolute time].
class ConditionedTrajectoryPerceiverIO(nn.Module):
    """The active Reacher DPF backbone: concat-state torque, zero external conditioning."""

    def __init__(self, num_input_channels: int, num_output_channels: int, state_dim: int,
                 num_latents: int = 256, num_latent_channels: int = 256, cond_dim: int = 256,
                 num_decoder_blocks: int = 4, num_heads: int = 8, dropout: float = 0.0):
        super().__init__()
        self.cond_dim = cond_dim
        self.encoder = ConditionedPerceiverEncoder(num_input_channels, num_latents, num_latent_channels, cond_dim, state_dim, 8, num_heads, dropout)
        self.decoder = ConditionedPerceiverDecoder(num_input_channels, num_latent_channels, num_output_channels, state_dim, cond_dim, num_heads, dropout, num_decoder_blocks)

    def forward(self, contexts: torch.Tensor, queries: torch.Tensor, torque: Optional[torch.Tensor] = None) -> torch.Tensor:
        B, T, _ = queries.shape
        global_cond = torch.zeros(B, 1, self.cond_dim, device=queries.device, dtype=queries.dtype)
        per_step_cond = torch.zeros(B, T, self.cond_dim, device=queries.device, dtype=queries.dtype)
        latents = self.encoder(contexts, global_cond)
        return self.decoder(latents, queries, per_step_cond)


class TrajectoryDPF(pl.LightningModule):
    """Raw-qpos Reacher DPF matching the shared concat-state shifted-tau checkpoint."""

    def __init__(self, qpos_dim: int, mom_dim: int, torque_dim: int, max_timesteps: int = 1000,
                 diffusion_steps: int = 1000, num_frequency_bands_for_diffusion: int = 64,
                 num_latents: int = 256, num_latent_channels: int = 256, cond_dim: int = 256,
                 num_decoder_blocks: int = 4,
                 trajectory_length_training_options: Tuple[int, ...] = (100, 200, 300, 400, 500, 600, 700, 800, 900, 1000),
                 lr: float = 1e-4, weight_decay: float = 1e-4, adam_beta1: float = 0.9, adam_beta2: float = 0.99,
                 warmup_steps: int = 1000, grad_warn_threshold: float = 10.0, use_fused_adamw: bool = False,
                 use_ema: bool = True, p_uncond: float = 0.1, encoder_cond_mode: str = 'none',
                 backbone: str = 'perceiverio', conditioning_mode: str = 'concat_torque_in_state',
                 query_context_mode: str = 'clean_prefix_noisy_suffix', token_layout: str = 'shifted_tau',
                 unconditional_tau_in_state: Optional[bool] = None, training_context_mode: Optional[str] = None,
                 dt: float = 0.001, data_dt: float = 0.001, xml_content: Optional[str] = None,
                 qpos_representation: str = RAW_QPOS,
                 qpos_min: Optional[torch.Tensor] = None, qpos_max: Optional[torch.Tensor] = None,
                 mom_min: Optional[torch.Tensor] = None, mom_max: Optional[torch.Tensor] = None,
                 torque_min: Optional[torch.Tensor] = None, torque_max: Optional[torch.Tensor] = None):
        super().__init__()
        if unconditional_tau_in_state is not None:
            conditioning_mode = 'concat_torque_in_state' if bool(unconditional_tau_in_state) else conditioning_mode
        if training_context_mode is not None and training_context_mode == 'shifted_future_context_cleanprefix':
            query_context_mode, token_layout, conditioning_mode = 'clean_prefix_noisy_suffix', 'shifted_tau', 'concat_torque_in_state'
        if (backbone, conditioning_mode, query_context_mode, token_layout, qpos_representation) != (
            'perceiverio', 'concat_torque_in_state', 'clean_prefix_noisy_suffix', 'shifted_tau', RAW_QPOS
        ):
            raise ValueError('This slim notebook supports only the shared raw-qpos Reacher concat-state shifted-tau Perceiver DPF config.')

        self.save_hyperparameters(ignore=['unconditional_tau_in_state', 'training_context_mode'])
        self.qpos_dim = qpos_dim
        self.raw_qpos_dim = raw_qpos_dim(qpos_dim, qpos_representation)
        self.mom_dim = mom_dim
        self.torque_dim = torque_dim
        self.qpos_representation = qpos_representation
        self.state_dim = qpos_dim + mom_dim + torque_dim
        self.max_timesteps = max_timesteps
        self.diffusion_steps = diffusion_steps
        self.trajectory_length_training_options = tuple(trajectory_length_training_options)
        self.lr = lr
        self.weight_decay = weight_decay
        self.adam_beta1 = adam_beta1
        self.adam_beta2 = adam_beta2
        self.warmup_steps = warmup_steps
        self.grad_warn_threshold = grad_warn_threshold
        self.use_fused_adamw = use_fused_adamw
        self.dt = dt
        self.data_dt = data_dt
        self.xml_content = xml_content
        self.conditioning_mode = conditioning_mode
        self.query_context_mode = query_context_mode
        self.token_layout = token_layout
        self.backbone = backbone
        self.unconditional_tau_in_state = True
        self.use_shifted_tau_tokens = True
        self.range_epsilon = DEFAULT_NORMALIZATION_RANGE_EPSILON

        self.fpe_diffusion = FourierPositionEncoding(input_shape=(diffusion_steps,), num_frequency_bands=num_frequency_bands_for_diffusion)
        self.diffusion_encoding_channels = self.fpe_diffusion.num_position_encoding_channels()
        self.num_temporal_frequency_bands = num_frequency_bands_for_diffusion // 2
        self.temporal_encoding_channels = self.num_temporal_frequency_bands * 2
        num_heads = 8
        raw_channels = self.state_dim + self.diffusion_encoding_channels + self.temporal_encoding_channels
        if raw_channels % num_heads != 0:
            self.temporal_encoding_channels += num_heads - (raw_channels % num_heads)
        num_input_channels = self.state_dim + self.diffusion_encoding_channels + self.temporal_encoding_channels

        with torch.no_grad():
            self.register_buffer('diffusion_encoding_table', self.fpe_diffusion(1)[0].contiguous(), persistent=True)
            self.register_buffer('temporal_encoding_table', self._get_temporal_encoding(max_timesteps, torch.device('cpu')).contiguous(), persistent=True)

        # DPF denoiser: predicts diffusion noise for every queried trajectory token.
        self.model = ConditionedTrajectoryPerceiverIO(
            num_input_channels=num_input_channels,
            num_output_channels=self.state_dim,
            state_dim=self.state_dim,
            num_latents=num_latents,
            num_latent_channels=num_latent_channels,
            cond_dim=cond_dim,
            num_decoder_blocks=num_decoder_blocks,
        )

        # Cosine diffusion schedule, matching the checkpoint configuration.
        s = 0.008
        t_vals = torch.linspace(0, diffusion_steps, diffusion_steps + 1, dtype=torch.float32)
        f = torch.cos(((t_vals / diffusion_steps + s) / (1.0 + s)) * math.pi / 2) ** 2
        alpha_bar = f / f[0]
        betas = torch.clamp(1.0 - (alpha_bar[1:] / alpha_bar[:-1]), min=1e-8, max=0.999)
        alphas = 1.0 - betas
        alpha_cumprod = torch.cumprod(alphas, dim=0)
        self.register_buffer('betas', betas)
        self.register_buffer('alphas', alphas)
        self.register_buffer('alpha_cumprod', alpha_cumprod)
        self.register_buffer('sqrt_alpha_cumprod', torch.sqrt(alpha_cumprod))
        self.register_buffer('sqrt_one_minus_alpha_cumprod', torch.sqrt(1.0 - alpha_cumprod))
        self._setup_normalization(qpos_min, qpos_max, mom_min, mom_max, torque_min, torque_max)
        self.ema = EMA(self.model, decay=0.9995) if use_ema else None
        self._ema_loaded = False

    def _get_temporal_encoding(self, T: int, device: torch.device) -> torch.Tensor:
        d = self.temporal_encoding_channels
        positions = torch.arange(T, device=device, dtype=torch.float32).unsqueeze(1)
        num_pairs = d // 2
        freqs = 1.0 / (10000.0 ** (2.0 * torch.arange(num_pairs, device=device, dtype=torch.float32) / d))
        angles = positions * freqs
        enc = torch.zeros(T, d, device=device)
        enc[:, :num_pairs] = torch.sin(angles)
        enc[:, num_pairs:2 * num_pairs] = torch.cos(angles)
        return enc

    def _setup_normalization(self, qpos_min, qpos_max, mom_min, mom_max, torque_min, torque_max):
        qpos_min = torch.zeros(self.qpos_dim) - 1 if qpos_min is None else qpos_min
        qpos_max = torch.ones(self.qpos_dim) if qpos_max is None else qpos_max
        mom_min = torch.zeros(self.mom_dim) - 1 if mom_min is None else mom_min
        mom_max = torch.ones(self.mom_dim) if mom_max is None else mom_max
        torque_min = torch.zeros(self.torque_dim) - 1 if torque_min is None else torque_min
        torque_max = torch.ones(self.torque_dim) if torque_max is None else torque_max
        state_min = torch.cat([qpos_min, mom_min, torque_min], dim=-1).float()
        state_max = torch.cat([qpos_max, mom_max, torque_max], dim=-1).float()
        state_max = torch.where((state_max - state_min) < self.range_epsilon, state_min + self.range_epsilon, state_max)
        torque_max = torch.where((torque_max - torque_min) < self.range_epsilon, torque_min + self.range_epsilon, torque_max)
        self.register_buffer('state_min', state_min)
        self.register_buffer('state_max', state_max)
        self.register_buffer('state_range', state_max - state_min)
        self.register_buffer('cond_min', torque_min.float())
        self.register_buffer('cond_max', torque_max.float())
        self.register_buffer('cond_range', (torque_max - torque_min).float())

    def normalize_state(self, state):
        return (state - self.state_min) / self.state_range * 2.0 - 1.0

    def denormalize_state(self, state):
        return (state + 1.0) / 2.0 * self.state_range + self.state_min

    def _shift_torque_sequence(self, torque: torch.Tensor) -> torch.Tensor:
        # Token at time t contains the torque applied before reaching state t.
        return torch.cat([torch.zeros_like(torque[:, :1, :]), torque[:, :-1, :]], dim=1)

    def _shifted_token_torque_to_rollout_torque(self, shifted_token_torque: torch.Tensor) -> torch.Tensor:
        return torch.cat([shifted_token_torque[:, 1:, :], shifted_token_torque[:, -1:, :]], dim=1)

    def build_tokens(self, state: torch.Tensor, diffusion_t: int, skip_normalize: bool = False, time_indices: Optional[torch.Tensor] = None) -> torch.Tensor:
        # Append diffusion-step and absolute-time encodings to each physical state token.
        B, T, _ = state.shape
        state_norm = state if skip_normalize else self.normalize_state(state)
        diffusion_vec = self.diffusion_encoding_table[diffusion_t - 1].to(device=state.device, dtype=state_norm.dtype)
        diffusion_enc = diffusion_vec.view(1, 1, -1).expand(B, T, -1)
        if time_indices is None:
            time_indices = torch.arange(T, device=state.device, dtype=torch.long)
        else:
            time_indices = time_indices.to(device=state.device, dtype=torch.long)
        max_idx = int(time_indices.max().item()) if time_indices.numel() else -1
        if max_idx < self.temporal_encoding_table.shape[0]:
            temporal_enc = self.temporal_encoding_table.index_select(0, time_indices).to(device=state.device, dtype=state_norm.dtype)
        else:
            temporal_enc = self._get_temporal_encoding(max_idx + 1, state.device).to(dtype=state_norm.dtype).index_select(0, time_indices)
        return torch.cat([state_norm, diffusion_enc, temporal_enc.unsqueeze(0).expand(B, -1, -1)], dim=-1)

    def _apply_state_noise_with_mask(self, clean_tokens: torch.Tensor, diffusion_t: int, state_noise_mask: torch.Tensor):
        if state_noise_mask.ndim == 1:
            state_noise_mask = state_noise_mask.unsqueeze(0).expand(clean_tokens.shape[0], -1)
        mask = state_noise_mask.to(device=clean_tokens.device, dtype=torch.bool).unsqueeze(-1)
        noise = torch.randn_like(clean_tokens[:, :, :self.state_dim])
        noisy_state = torch.where(
            mask,
            self.sqrt_alpha_cumprod[diffusion_t - 1] * clean_tokens[:, :, :self.state_dim] +
            self.sqrt_one_minus_alpha_cumprod[diffusion_t - 1] * noise,
            clean_tokens[:, :, :self.state_dim],
        )
        noisy_tokens = clean_tokens.clone()
        noisy_tokens[:, :, :self.state_dim] = noisy_state
        return noisy_tokens, noise * mask.to(dtype=noise.dtype)

    def _sample_subset_indices(self, seq_len: int, subset_len: int, device: torch.device):
        idx = torch.randperm(seq_len, device=device)[:max(1, min(int(subset_len), int(seq_len)))]
        return torch.sort(idx).values

    def _build_clean_prefix_noisy_suffix_views(self, state: torch.Tensor, diffusion_t: int, time_indices: torch.Tensor):
        _, T, _ = state.shape
        clean_tokens = self.build_tokens(state, diffusion_t, time_indices=time_indices)
        # Training objective: keep a clean prefix/context and ask the model to denoise a suffix.
        suffix_start = torch.randint(1, T, (1,), device=state.device).item() if T > 1 else 0
        loss_mask = torch.zeros(T, device=state.device, dtype=torch.bool)
        loss_mask[suffix_start:] = True
        noisy_queries, noise = self._apply_state_noise_with_mask(clean_tokens, diffusion_t, loss_mask)
        context_idx = self._sample_subset_indices(T, torch.randint(1, T + 1, (1,), device=state.device).item(), state.device)
        return noisy_queries.index_select(1, context_idx), noisy_queries, noise, loss_mask, int(context_idx.numel()), T

    def _compute_denoise_loss(self, predictions: torch.Tensor, noise_target: torch.Tensor, loss_mask: torch.Tensor):
        weight = loss_mask.unsqueeze(0).expand(predictions.shape[0], -1).to(predictions.device, predictions.dtype).unsqueeze(-1)
        return (((predictions - noise_target) ** 2) * weight).sum() / (weight.sum() * predictions.shape[-1]).clamp_min(1.0)

    def training_step(self, batch, batch_idx):
        qpos, mom, torque = batch['seq_qpos'], batch['seq_mom'], batch['seq_torque']
        state = torch.cat([qpos, mom, self._shift_torque_sequence(torque)], dim=-1)
        _, T, _ = state.shape
        T_train = min(self.trajectory_length_training_options[torch.randint(0, len(self.trajectory_length_training_options), (1,)).item()], T)
        start = torch.randint(0, T - T_train + 1, (1,)).item() if T > T_train else 0
        # Random windows teach the same model to complete different rollout horizons.
        state = state[:, start:start + T_train, :]
        time_indices = torch.arange(start, start + T_train, device=state.device, dtype=torch.long)
        diffusion_t = torch.randint(1, self.diffusion_steps + 1, (1,)).item()
        contexts, queries, noise_target, loss_mask, num_context, num_query = self._build_clean_prefix_noisy_suffix_views(state, diffusion_t, time_indices)
        loss = self._compute_denoise_loss(self.model(contexts, queries, torque=None), noise_target, loss_mask)
        self.log('train_loss', loss, prog_bar=True, on_step=True, on_epoch=False, sync_dist=True)
        self.log('T_train', float(T_train), on_step=True, on_epoch=False, sync_dist=True)
        self.log('num_context', float(num_context), on_step=True, on_epoch=False, sync_dist=True)
        self.log('num_query', float(num_query), on_step=True, on_epoch=False, sync_dist=True)
        return loss

    def validation_step(self, batch, batch_idx):
        qpos, mom, torque = batch['seq_qpos'], batch['seq_mom'], batch['seq_torque']
        state = torch.cat([qpos, mom, self._shift_torque_sequence(torque)], dim=-1)
        T = state.shape[1]
        diffusion_t = torch.randint(1, self.diffusion_steps + 1, (1,)).item()
        time_indices = torch.arange(T, device=state.device, dtype=torch.long)
        contexts, queries, noise_target, loss_mask, _, _ = self._build_clean_prefix_noisy_suffix_views(state, diffusion_t, time_indices)
        loss = self._compute_denoise_loss(self.model(contexts, queries, torque=None), noise_target, loss_mask)
        self.log('val_loss', loss, prog_bar=True, on_step=False, on_epoch=True, sync_dist=True)
        return loss

    def configure_optimizers(self):
        optimizer = torch.optim.AdamW(self.model.parameters(), lr=self.lr, weight_decay=self.weight_decay, betas=(self.adam_beta1, self.adam_beta2))
        total_steps = self.trainer.estimated_stepping_batches
        warmup_steps = min(int(self.warmup_steps), int(total_steps)) if self.warmup_steps > 0 else min(1000, total_steps // 10)
        def lr_lambda(step):
            if step < warmup_steps:
                return float(step) / float(max(1, warmup_steps))
            progress = float(step - warmup_steps) / float(max(1, total_steps - warmup_steps))
            return max(0.1, 0.5 * (1.0 + math.cos(math.pi * progress)))
        return {'optimizer': optimizer, 'lr_scheduler': {'scheduler': torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda), 'interval': 'step'}}

    def on_save_checkpoint(self, checkpoint):
        if self.ema is not None and self.ema.shadow:
            checkpoint['ema_decay'] = self.ema.decay
            checkpoint['ema_shadow'] = {name: tensor.detach().cpu() for name, tensor in self.ema.shadow.items()}

    def on_load_checkpoint(self, checkpoint):
        ema_shadow = checkpoint.get('ema_shadow')
        if ema_shadow is not None and self.ema is not None:
            for name, tensor in ema_shadow.items():
                if name in self.ema.shadow:
                    self.ema.shadow[name] = tensor.to(dtype=self.ema.shadow[name].dtype)
            self._ema_loaded = True

    def load_state_dict(self, state_dict, strict=True):
        result = super().load_state_dict(state_dict, strict=False)
        important_missing = [k for k in result.missing_keys if not k.startswith('model.torque_conditioner')]
        ignored_unexpected = [k for k in result.unexpected_keys if k.startswith(('model.c0', 'model.torque_conditioner', 'model.state_conditioner', 'model.interaction_mlp'))]
        other_unexpected = [k for k in result.unexpected_keys if k not in ignored_unexpected]
        if important_missing:
            raise RuntimeError(f'Missing required checkpoint keys: {important_missing[:10]}')
        if other_unexpected:
            raise RuntimeError(f'Unexpected checkpoint keys outside the known unused conditioning modules: {other_unexpected[:10]}')
        return result

    def on_fit_start(self):
        if self.ema is not None:
            for name, p in self.model.named_parameters():
                if p.requires_grad and name in self.ema.shadow:
                    self.ema.shadow[name] = self.ema.shadow[name].to(device=p.device, dtype=p.dtype)

    def on_before_optimizer_step(self, optimizer):
        total_norm = 0.0
        for p in self.model.parameters():
            if p.grad is not None:
                total_norm += p.grad.data.norm(2).item() ** 2
        total_norm = total_norm ** 0.5
        self.log('grad_norm', total_norm, on_step=True, on_epoch=False, sync_dist=True)

    def on_train_batch_end(self, outputs, batch, batch_idx):
        if self.ema is not None:
            self.ema.update(self.model)



    def _expand_sampling_tensor(self, tensor, num_samples, trajectory_length, expected_last_dim, name):
        if tensor is None:
            raise ValueError(f'{name} is required for Reacher prefix completion.')
        if tensor.ndim != 3 or tensor.shape[-1] != expected_last_dim or tensor.shape[1] < trajectory_length:
            raise ValueError(f'{name} must have shape [B,T,{expected_last_dim}] with T >= {trajectory_length}; got {tuple(tensor.shape)}')
        if tensor.shape[0] == 1 and num_samples > 1:
            tensor = tensor.expand(num_samples, -1, -1)
        elif tensor.shape[0] != num_samples:
            raise ValueError(f'{name} batch dim must be 1 or {num_samples}; got {tensor.shape[0]}')
        return tensor[:, :trajectory_length, :].to(device=self.device, dtype=torch.float32)

    def _expand_target_xy(self, target_xy: Optional[torch.Tensor], num_samples: int) -> Optional[torch.Tensor]:
        if target_xy is None:
            return None
        if target_xy.ndim == 1:
            target_xy = target_xy.unsqueeze(0)
        if target_xy.shape[-1] != 2:
            raise ValueError(f'target_xy must have last dim 2, got {tuple(target_xy.shape)}')
        if target_xy.shape[0] == 1 and num_samples > 1:
            target_xy = target_xy.expand(num_samples, -1)
        elif target_xy.shape[0] != num_samples:
            raise ValueError(f'target_xy batch dim must be 1 or {num_samples}; got {target_xy.shape[0]}')
        return target_xy.to(device=self.device, dtype=torch.float32)

    def _apply_prefix_constraint(self, x: torch.Tensor, observed_prefix_state_norm: torch.Tensor, prefix_len: int) -> torch.Tensor:
        """Keep only the observed prefix fixed; the suffix is generated freely."""
        if prefix_len <= 0:
            return x
        x[:, :prefix_len, :] = observed_prefix_state_norm[:, :prefix_len, :]
        return x

    def _predict_x0(self, x_t, eps, a_bar_t, percentile: float = 0.995):
        x0 = (x_t - torch.sqrt(1.0 - a_bar_t) * eps) / torch.sqrt(a_bar_t)
        flat = x0.reshape(x0.shape[0], -1).abs()
        k = max(1, min(int(percentile * flat.shape[1]), flat.shape[1] - 1))
        threshold = flat.kthvalue(k, dim=1, keepdim=True).values.clamp_min(1.0).unsqueeze(-1)
        return torch.clamp(x0 / threshold, -1.0, 1.0)

    def _reacher_fingertip_xy_torch(self, qpos_raw: torch.Tensor) -> torch.Tensor:
        q0 = qpos_raw[..., 0]
        q1 = qpos_raw[..., 1]
        return torch.stack([
            0.10 * torch.cos(q0) + 0.11 * torch.cos(q0 + q1),
            0.10 * torch.sin(q0) + 0.11 * torch.sin(q0 + q1),
        ], dim=-1)

    def _run_one_step_target_guidance(self, state_phys: torch.Tensor, target_xy: torch.Tensor, prefix_len: int,
                                      alpha_q: float, norm: str = 'l2', time_power: float = 2.0,
                                      normalize_grad: bool = True, use_time_weights: bool = True) -> torch.Tensor:
        if alpha_q <= 0 or target_xy is None:
            return state_phys
        state_var = state_phys.detach().requires_grad_(True)
        qpos = state_var[:, :, :self.qpos_dim]
        suffix_start = max(0, min(int(prefix_len), qpos.shape[1] - 1))
        ee = self._reacher_fingertip_xy_torch(qpos[:, suffix_start:, :])
        err = ee - target_xy[:, None, :]
        abs_err = torch.abs(err)
        if norm == 'l2':
            per_step = (err ** 2).sum(dim=-1)
        elif norm == 'l1':
            per_step = abs_err.sum(dim=-1)
        elif norm in {'linf', 'l_inf', 'l-infinity'}:
            per_step = abs_err.amax(dim=-1)
        else:
            raise ValueError("target guidance norm must be one of {'l1', 'l2', 'linf'}")
        if use_time_weights:
            weights = torch.linspace(1.0 / max(per_step.shape[1], 1), 1.0, per_step.shape[1], device=per_step.device, dtype=per_step.dtype)
            weights = weights.pow(float(time_power))
            weights = weights / weights.sum().clamp_min(1e-8)
            loss = (per_step * weights.view(1, -1)).sum(dim=1).mean()
        else:
            loss = per_step.mean()
        grad_q = torch.autograd.grad(loss, qpos, retain_graph=False, create_graph=False)[0]
        if normalize_grad:
            grad_q = grad_q / torch.sqrt(torch.mean(grad_q.detach() ** 2, dim=(1, 2), keepdim=True) + 1e-8)
        guided = state_phys.clone()
        guided[:, :, :self.qpos_dim] = (qpos - float(alpha_q) * grad_q).detach()
        return guided

    def sample_trajectories(self, num_samples: int, trajectory_length: int, num_diffusion_steps: int = 20,
                            prefix_len: Optional[int] = None, observed_qpos=None, observed_mom=None,
                            observed_torque=None, time_indices=None, use_ema: bool = False,
                            sampler: str = 'ddim', initial_noise: Optional[torch.Tensor] = None,
                            hnn: Optional[nn.Module] = None, alpha_q: float = 1e-4, alpha_p: float = 1e-4,
                            guidance_trust_lambda: float = 0.0, guidance_normalize_grad: bool = True,
                            guidance_joint_update: bool = False, target_xy: Optional[torch.Tensor] = None,
                            target_guidance_alpha: float = 0.0, target_guidance_time_power: float = 2.0,
                            target_guidance_normalize_grad: bool = True, target_guidance_norm: str = 'l2',
                            target_guidance_use_time_weights: bool = True, guidance_order: str = 'hnn_then_target',
                            **unused_kwargs):
        """
        Reacher prefix completion with optional HNN and target-space guidance.

        Base DPF path: query = [clean prefix + noisy/generated suffix], context = a random subset of the query.
        HNN guidance: one differentiable step on the generated x0 trajectory to reduce HNN one-step physics residual.
        Target guidance: one differentiable step on qpos to reduce terminal/late-suffix EE error under L1/L2/Linf.
        """
        if sampler != 'ddim':
            raise ValueError('This notebook keeps only the DDIM prefix-completion sampler.')
        if guidance_order not in {'hnn_then_target', 'target_then_hnn'}:
            raise ValueError("guidance_order must be 'hnn_then_target' or 'target_then_hnn'")
        prefix_len = 1 if prefix_len is None else max(1, min(int(prefix_len), trajectory_length - 1))
        qpos = self._expand_sampling_tensor(observed_qpos, num_samples, trajectory_length, self.qpos_dim, 'observed_qpos')
        mom = self._expand_sampling_tensor(observed_mom, num_samples, trajectory_length, self.mom_dim, 'observed_mom')
        tau = self._expand_sampling_tensor(observed_torque, num_samples, trajectory_length, self.torque_dim, 'observed_torque')
        target_xy = self._expand_target_xy(target_xy, num_samples)
        observed_state = torch.cat([qpos, mom, self._shift_torque_sequence(tau)], dim=-1)
        observed_state_norm = self.normalize_state(observed_state)

        if time_indices is not None:
            time_indices = time_indices.to(device=self.device, dtype=torch.long)

        # Inference mirrors training: context is a subset of the clean-prefix/noisy-suffix query.
        context_len = max(1, min(prefix_len, trajectory_length))
        context_idx = torch.randperm(trajectory_length, device=self.device)[:context_len].sort().values

        was_training = self.model.training
        self.model.eval()
        if hnn is not None:
            hnn.eval()
        if use_ema and self.ema is not None:
            self.ema.store(self.model)
            self.ema.copy_to(self.model)
        try:
            # Start from Gaussian suffix noise; prefix entries are clamped after every update.
            x = initial_noise.to(self.device) if initial_noise is not None else torch.randn(num_samples, trajectory_length, self.state_dim, device=self.device)
            x = self._apply_prefix_constraint(x, observed_state_norm, prefix_len)
            ts = torch.linspace(self.diffusion_steps - 1, 0, steps=num_diffusion_steps, device=self.device, dtype=torch.long)
            for i, t in enumerate(tqdm(ts, total=len(ts), desc='Sampling')):
                x = self._apply_prefix_constraint(x, observed_state_norm, prefix_len)
                t_int = int(t.item())
                with torch.no_grad():
                    # Predict epsilon for the full candidate trajectory at the current diffusion step.
                    queries = self.build_tokens(x, t_int + 1, skip_normalize=True, time_indices=time_indices)
                    contexts = queries.index_select(1, context_idx)
                    eps = self.model(contexts, queries, torque=None)
                a_bar_t = self.alpha_cumprod[t_int].clamp(min=1e-6, max=1.0)
                a_bar_prev = self.alpha_cumprod[int(ts[i + 1].item())].clamp(min=1e-6, max=1.0) if i < len(ts) - 1 else a_bar_t.new_tensor(1.0)
                x0 = self._predict_x0(x, eps, a_bar_t)

                # Guidance is applied to predicted x0 in physical units, then re-normalized.
                if hnn is not None or (target_xy is not None and target_guidance_alpha > 0):
                    x0_phys = self.denormalize_state(x0)
                    noise_step_scale = torch.sqrt(torch.clamp(1.0 - a_bar_t, min=1e-6)).item()

                    def apply_hnn(current_phys):
                        # HNN guidance updates q,p only; generated torque tokens are kept fixed.
                        if hnn is None:
                            return current_phys
                        torque_tokens = current_phys[:, :, self.qpos_dim + self.mom_dim:self.qpos_dim + self.mom_dim + self.torque_dim]
                        guided_qp = run_one_step_guidance_hnn(
                            current_phys, torque_tokens, self.qpos_dim, self.mom_dim, self.data_dt, hnn,
                            qpos_representation=self.qpos_representation,
                            alpha_q=float(alpha_q) * noise_step_scale,
                            alpha_p=float(alpha_p) * noise_step_scale,
                            guidance_trust_lambda=float(guidance_trust_lambda),
                            guidance_normalize_grad=bool(guidance_normalize_grad),
                            guidance_joint_update=bool(guidance_joint_update),
                        )
                        return torch.cat([guided_qp, torque_tokens.detach()], dim=-1)

                    def apply_target(current_phys):
                        # Target guidance updates qpos to reduce end-effector distance to target_xy.
                        return self._run_one_step_target_guidance(
                            current_phys, target_xy=target_xy, prefix_len=prefix_len,
                            alpha_q=float(target_guidance_alpha), norm=str(target_guidance_norm),
                            time_power=float(target_guidance_time_power),
                            normalize_grad=bool(target_guidance_normalize_grad),
                            use_time_weights=bool(target_guidance_use_time_weights),
                        )

                    # The order is exposed in presets because the second guidance step sees the first step's update.
                    if guidance_order == 'target_then_hnn':
                        x0_phys = apply_hnn(apply_target(x0_phys))
                    else:
                        x0_phys = apply_target(apply_hnn(x0_phys))
                    x0 = self.normalize_state(x0_phys).detach()
                    x0 = self._apply_prefix_constraint(x0, observed_state_norm, prefix_len)

                x = torch.sqrt(a_bar_prev) * x0 + torch.sqrt(1.0 - a_bar_prev) * eps
            x = self._apply_prefix_constraint(x, observed_state_norm, prefix_len)
            state = self.denormalize_state(x)
            shifted_tau = state[:, :, self.qpos_dim + self.mom_dim:self.qpos_dim + self.mom_dim + self.torque_dim]
            tau_rollout = self._shifted_token_torque_to_rollout_torque(shifted_tau)
            return state, tau_rollout
        finally:
            if use_ema and self.ema is not None:
                self.ema.restore(self.model)
            if was_training:
                self.model.train()



print('Loaded slim Reacher-only DPF classes: TrajectoryDPFCached, TrajectoryDPF')


## 4. Self-Contained HNN Model, Training, and Guidance

This section embeds the structured Reacher HNN used by the guided DPF variants. It can load the shared HNN checkpoint for guidance, or train a new HNN from the same HDF5 dataset.


In [ ]:

from dataclasses import dataclass


def infer_hnn_torque_alignment(h5_attrs) -> str:
    # Older and newer datasets store torque at slightly different transition alignments.
    state_alignment = h5_attrs.get('state_alignment', None)
    torque_alignment = h5_attrs.get('torque_alignment', None)
    derivative_alignment = h5_attrs.get('derivative_alignment', None)
    for name, value in [('state', state_alignment), ('torque', torque_alignment), ('derivative', derivative_alignment)]:
        if isinstance(value, bytes):
            if name == 'state':
                state_alignment = value.decode()
            elif name == 'torque':
                torque_alignment = value.decode()
            else:
                derivative_alignment = value.decode()
    if state_alignment == 'pre_step' and torque_alignment == 'interval_mean' and derivative_alignment == 'forward_difference':
        return 'legacy'
    return 'transition_next'


class TrajectoryHNNCached(Dataset):
    """Flattened Reacher HNN supervision: (q, p, torque) -> (qdot, pdot)."""

    def __init__(self, h5_path: str, trajectory_length: int = 1000, torque_alignment: str = 'auto'):
        super().__init__()
        if torque_alignment not in {'auto', 'legacy', 'transition_next'}:
            raise ValueError("torque_alignment must be 'auto', 'legacy', or 'transition_next'")
        print(f'Loading HNN dataset into memory from {h5_path}...')
        with h5py.File(h5_path, 'r') as f:
            self.num_traj = int(f.attrs['num_trajectories'])
            self.num_steps = int(f.attrs['num_steps'])
            self.dt = float(f.attrs.get('dt', 0.001))
            self.resolved_torque_alignment = infer_hnn_torque_alignment(f.attrs) if torque_alignment == 'auto' else torque_alignment
            qpos, qvel, qacc, mom, mom_dot, torque = [], [], [], [], [], []
            for i in range(self.num_traj):
                traj = f[f'traj_{i}']
                sl = slice(0, trajectory_length)
                q = traj['seq_qpos'][sl]
                v = traj['seq_qvel'][sl]
                a = traj['seq_qacc'][sl]
                p = traj['seq_mom'][sl]
                pdot = traj['seq_mom_dot'][sl]
                tau = traj['seq_torque'][sl]
                if self.resolved_torque_alignment == 'transition_next':
                    q, v, a, p, pdot, tau = q[:-1], v[:-1], a[:-1], p[:-1], pdot[:-1], tau[1:]
                qpos.append(q); qvel.append(v); qacc.append(a); mom.append(p); mom_dot.append(pdot); torque.append(tau)
        self.qpos = torch.from_numpy(np.concatenate(qpos, axis=0).astype(np.float32))
        self.qvel = torch.from_numpy(np.concatenate(qvel, axis=0).astype(np.float32))
        self.qacc = torch.from_numpy(np.concatenate(qacc, axis=0).astype(np.float32))
        self.mom = torch.from_numpy(np.concatenate(mom, axis=0).astype(np.float32))
        self.mom_dot = torch.from_numpy(np.concatenate(mom_dot, axis=0).astype(np.float32))
        self.torque = torch.from_numpy(np.concatenate(torque, axis=0).astype(np.float32))
        print(f'HNN torque alignment resolved={self.resolved_torque_alignment}; samples={len(self)}')

    def __len__(self):
        return int(self.qpos.shape[0])

    def __getitem__(self, index):
        return {
            'qpos': self.qpos[index],
            'qvel': self.qvel[index],
            'qacc': self.qacc[index],
            'mom': self.mom[index],
            'mom_dot': self.mom_dot[index],
            'torque': self.torque[index],
        }


# Structured HNN learns a Hamiltonian H(q,p)=T(q,p)+V(q), then uses gradients for dynamics.
class StructuredHNN(nn.Module):
    """Mechanical HNN: T(q,p)=0.5*p^T M^{-1}(q)p and learned V(q)."""

    def __init__(self, coordinate_dim, momenta_dim, hidden_dim=256, num_layers=4):
        super().__init__()
        self.dim = coordinate_dim
        self.num_chol_params = coordinate_dim * (coordinate_dim + 1) // 2
        trig_input_dim = 3 * coordinate_dim
        def mlp(out_dim):
            layers = [nn.Linear(trig_input_dim, hidden_dim), nn.SiLU()]
            for _ in range(num_layers - 2):
                layers += [nn.Linear(hidden_dim, hidden_dim), nn.SiLU()]
            layers.append(nn.Linear(hidden_dim, out_dim))
            return nn.Sequential(*layers)
        self.cholesky_net = mlp(self.num_chol_params)
        self.potential_net = mlp(1)
        self.register_buffer('tril_rows', torch.tril_indices(coordinate_dim, coordinate_dim)[0])
        self.register_buffer('tril_cols', torch.tril_indices(coordinate_dim, coordinate_dim)[1])
        self.register_buffer('diag_idx', torch.arange(coordinate_dim))

    def _trig_features(self, q):
        return torch.cat([q, torch.sin(q), torch.cos(q)], dim=-1)

    def _get_cholesky(self, q):
        raw = self.cholesky_net(self._trig_features(q))
        B = q.shape[0]
        L = torch.zeros(B, self.dim, self.dim, device=q.device, dtype=q.dtype)
        L[:, self.tril_rows, self.tril_cols] = raw
        L[:, self.diag_idx, self.diag_idx] = F.softplus(L[:, self.diag_idx, self.diag_idx]) + 1e-4
        return L

    def forward(self, p, q):
        L = self._get_cholesky(q)
        Ltp = torch.bmm(L.transpose(1, 2), p.unsqueeze(-1))
        T = 0.5 * (Ltp.squeeze(-1) ** 2).sum(dim=-1, keepdim=True)
        V = self.potential_net(self._trig_features(q))
        return T + V


class HNNWrapper(pl.LightningModule):
    def __init__(self, coordinate_dim, momenta_dim, use_torque=True, predict_torque=False,
                 qvel_var=1.0, mom_dot_var=1.0, q_std=1.0, p_std=1.0, eps: float = 1e-8,
                 model_type='structured', hidden_dim=256, num_layers=4, lr=3e-4, torque_gain=1.0):
        super().__init__()
        self.save_hyperparameters()
        if model_type != 'structured':
            raise ValueError('This self-contained Reacher package includes the structured HNN used by the checkpoint.')
        self.model = StructuredHNN(coordinate_dim, momenta_dim, hidden_dim=hidden_dim, num_layers=num_layers)
        self.use_torque = bool(use_torque)
        self.predict_torque = bool(predict_torque)
        self.register_buffer('eps', torch.tensor(float(eps)))
        self.register_buffer('qvel_var', torch.as_tensor(qvel_var, dtype=torch.float32))
        self.register_buffer('mom_dot_var', torch.as_tensor(mom_dot_var, dtype=torch.float32))
        self.register_buffer('q_std', torch.as_tensor(q_std, dtype=torch.float32))
        self.register_buffer('p_std', torch.as_tensor(p_std, dtype=torch.float32))
        self.register_buffer('torque_gain', torch.as_tensor(torque_gain, dtype=torch.float32))

    def _scale_torque(self, torque):
        return None if torque is None else torque * self.torque_gain

    def forward(self, p, q):
        return self.model(p / (self.p_std + self.eps), q / (self.q_std + self.eps))

    def calculate_loss(self, p, q, dqdt_target, dpdt_target, torque_target=None, qacc_target=None, create_graph=True):
        # Hamilton's equations: qdot=dH/dp and pdot=-dH/dq+tau.
        with torch.inference_mode(False):
            with torch.set_grad_enabled(True):
                p_raw = p.detach().requires_grad_(True)
                q_raw = q.detach().requires_grad_(True)
                H = self(p_raw, q_raw)
                dH_dp, dH_dq = torch.autograd.grad(H.sum(), (p_raw, q_raw), create_graph=create_graph)
                dqdt_pred = dH_dp
                dpdt_pred = -dH_dq
        if self.use_torque:
            dpdt_pred = dpdt_pred + self._scale_torque(torque_target)
        loss_dqdt = torch.mean((dqdt_target - dqdt_pred) ** 2 / (self.qvel_var + self.eps))
        loss_dpdt = torch.mean((dpdt_target - dpdt_pred) ** 2 / (self.mom_dot_var + self.eps))
        return loss_dqdt + loss_dpdt

    def training_step(self, batch, batch_idx):
        loss = self.calculate_loss(batch['mom'], batch['qpos'], batch['qvel'], batch['mom_dot'], batch['torque'])
        self.log('train_loss', loss, prog_bar=True)
        return loss

    def validation_step(self, batch, batch_idx):
        loss = self.calculate_loss(batch['mom'], batch['qpos'], batch['qvel'], batch['mom_dot'], batch['torque'], create_graph=False)
        self.log('val_loss', loss, prog_bar=True, sync_dist=True)
        return loss

    def configure_optimizers(self):
        optimizer = torch.optim.AdamW(self.parameters(), lr=float(self.hparams.get('lr', 3e-4)), weight_decay=0.0)
        total_steps = int(self.trainer.estimated_stepping_batches)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(total_steps, 1), eta_min=1e-6)
        return {'optimizer': optimizer, 'lr_scheduler': {'scheduler': scheduler, 'interval': 'step'}}


def compute_hnn_physics_energy(seq_qpos, seq_mom, seq_torque, hnn, dt, qpos_representation='raw'):
    # Measures whether adjacent generated states satisfy the HNN one-step dynamics.
    B, T, _ = seq_mom.shape
    q_t = seq_qpos[:, :-1]
    p_t = seq_mom[:, :-1]
    tau_t = seq_torque[:, :-1]
    q_next = seq_qpos[:, 1:]
    p_next = seq_mom[:, 1:]
    p_flat = p_t.reshape(-1, p_t.shape[-1]).detach().requires_grad_(True)
    q_flat = q_t.reshape(-1, q_t.shape[-1]).detach().requires_grad_(True)
    H = hnn(p_flat, q_flat)
    dH_dp, dH_dq = torch.autograd.grad(H.sum(), (p_flat, q_flat), create_graph=True)
    dH_dp = dH_dp.reshape(B, T - 1, -1)
    dH_dq = dH_dq.reshape(B, T - 1, -1)
    q_pred = q_t + dt * dH_dp
    p_pred = p_t + dt * (-dH_dq + tau_t)
    return (((q_next - q_pred) ** 2).sum(dim=-1) + ((p_next - p_pred) ** 2).sum(dim=-1)).mean()


def run_one_step_guidance_hnn(x, seq_torque, qpos_dim, mom_dim, dt, hnn, qpos_representation='raw',
                              alpha_q=1e-4, alpha_p=1e-4, guidance_trust_lambda=0.0,
                              guidance_normalize_grad=True, guidance_joint_update=False):
    seq_qpos = x[:, :, :qpos_dim].clone().requires_grad_(True)
    seq_mom = x[:, :, qpos_dim:qpos_dim + mom_dim].clone().requires_grad_(True)
    q_ref = seq_qpos.detach()
    p_ref = seq_mom.detach()
    # Take one gradient step that reduces the HNN physics residual of the generated trajectory.
    energy = compute_hnn_physics_energy(seq_qpos, seq_mom, seq_torque, hnn, dt, qpos_representation=qpos_representation)
    if guidance_trust_lambda > 0:
        energy = energy + guidance_trust_lambda * (((seq_qpos - q_ref) ** 2).mean() + ((seq_mom - p_ref) ** 2).mean())
    grad_q, grad_p = torch.autograd.grad(energy, [seq_qpos, seq_mom])
    eps = 1e-12
    if guidance_joint_update and guidance_normalize_grad:
        norm = torch.cat([grad_q, grad_p], dim=-1).flatten(1).norm(dim=1, keepdim=True).view(-1, 1, 1).clamp_min(eps)
        grad_q, grad_p = grad_q / norm, grad_p / norm
    elif guidance_normalize_grad:
        grad_q = grad_q / grad_q.flatten(1).norm(dim=1, keepdim=True).view(-1, 1, 1).clamp_min(eps)
        grad_p = grad_p / grad_p.flatten(1).norm(dim=1, keepdim=True).view(-1, 1, 1).clamp_min(eps)
    q_new = seq_qpos - float(alpha_q) * grad_q
    p_new = seq_mom - float(alpha_p) * grad_p
    return torch.cat([q_new.detach(), p_new.detach()], dim=-1)


@dataclass
class ReacherHNNTrainConfig:
    train_h5: Path = TRAIN_H5
    val_h5: Path = VAL_H5
    checkpoint_dir: Path = HNN_CKPT_DIR
    epochs: int = 1000
    batch_size: int = 2048
    num_workers: int = 8
    hidden_dim: int = 256
    num_layers: int = 4
    lr: float = 1.5e-4
    torque_gain: tuple = (200.0, 200.0)
    torque_alignment: str = 'auto'  # auto keeps the notebook compatible with copied H5 metadata
    max_train_batches: int = 2000
    limit_val_batches: float = 0.05


HNN_TRAIN_CFG = ReacherHNNTrainConfig()
print(HNN_TRAIN_CFG)


def make_reacher_hnn_dataloaders(cfg: ReacherHNNTrainConfig):
    train_data = TrajectoryHNNCached(str(cfg.train_h5), trajectory_length=1000, torque_alignment=cfg.torque_alignment)
    val_data = TrajectoryHNNCached(str(cfg.val_h5), trajectory_length=1000, torque_alignment=cfg.torque_alignment)
    train_loader = DataLoader(train_data, batch_size=cfg.batch_size, shuffle=True, num_workers=cfg.num_workers, pin_memory=torch.cuda.is_available())
    val_loader = DataLoader(val_data, batch_size=cfg.batch_size, shuffle=False, num_workers=cfg.num_workers, pin_memory=torch.cuda.is_available())
    return train_data, val_data, train_loader, val_loader


def build_reacher_hnn_model(cfg: ReacherHNNTrainConfig, train_data: TrajectoryHNNCached):
    dim = int(train_data.qpos.shape[-1])
    return HNNWrapper(
        dim, dim, use_torque=True, predict_torque=False,
        qvel_var=train_data.qvel.var(dim=0, unbiased=False),
        mom_dot_var=train_data.mom_dot.var(dim=0, unbiased=False),
        q_std=train_data.qpos.std(dim=0, unbiased=False),
        p_std=train_data.mom.std(dim=0, unbiased=False),
        model_type='structured', hidden_dim=cfg.hidden_dim, num_layers=cfg.num_layers,
        lr=cfg.lr, torque_gain=np.asarray(cfg.torque_gain, dtype=np.float32),
    )


def train_reacher_hnn(cfg: ReacherHNNTrainConfig = HNN_TRAIN_CFG):
    cfg.checkpoint_dir.mkdir(parents=True, exist_ok=True)
    train_data, val_data, train_loader, val_loader = make_reacher_hnn_dataloaders(cfg)
    hnn = build_reacher_hnn_model(cfg, train_data)
    checkpoint_callback = ModelCheckpoint(
        dirpath=str(cfg.checkpoint_dir),
        filename='StructuredHNN-ReacherExploration-IID-epoch-{epoch}',
        every_n_epochs=50,
        save_top_k=-1,
    )
    trainer = pl.Trainer(
        max_epochs=cfg.epochs,
        accelerator='gpu' if torch.cuda.is_available() else 'cpu',
        devices=[0] if torch.cuda.is_available() else 1,
        callbacks=[checkpoint_callback],
        logger=False,
        precision=32,
        gradient_clip_val=1.0,
        gradient_clip_algorithm='norm',
        limit_train_batches=cfg.max_train_batches,
        limit_val_batches=cfg.limit_val_batches,
        log_every_n_steps=50,
    )
    trainer.fit(hnn, train_loader, val_loader)
    return hnn, trainer


print('Loaded self-contained HNN classes/guidance: TrajectoryHNNCached, HNNWrapper, run_one_step_guidance_hnn')


## 5. DPF Training Workflow

The function below is the actual training workflow. It is left uncalled by default so opening the notebook does not accidentally start a long run.


In [ ]:
from dataclasses import dataclass


@dataclass
class ReacherDPFTrainConfig:
    train_h5: Path = TRAIN_H5
    val_h5: Path = VAL_H5
    checkpoint_dir: Path = CKPT_DIR
    devices: tuple = (0,)  # GPU ids used by PyTorch Lightning when CUDA is available
    epochs: int = 3000
    batch_size: int = 160
    num_workers: int = 16
    lr: float = 1e-4
    weight_decay: float = 1e-4
    adam_beta1: float = 0.9
    adam_beta2: float = 0.99
    warmup_steps: int = 1000
    grad_clip_val: float = 1.0
    num_decoder_blocks: int = 4
    num_latents: int = 256
    num_latent_channels: int = 256
    diffusion_steps: int = 1000
    backbone: str = 'perceiverio'
    conditioning_mode: str = 'concat_torque_in_state'
    query_context_mode: str = 'clean_prefix_noisy_suffix'
    token_layout: str = 'shifted_tau'
    qpos_representation_override: str = 'raw'  # this slim notebook supports raw Reacher qpos only
    checkpoint_every_n_epochs: int = 10
    check_val_every_n_epoch: int = 1
    num_sanity_val_steps: int = 2
    max_trajectories: int = 0  # set small, e.g. 128, for a quick smoke run
    disable_wandb: bool = True


TRAIN_CFG = ReacherDPFTrainConfig()
print(TRAIN_CFG)


def make_reacher_dpf_dataloaders(cfg: ReacherDPFTrainConfig):
    qpos_override = None if cfg.qpos_representation_override == 'auto' else cfg.qpos_representation_override
    # train_dataset_full is kept for normalization/stat metadata even if max_trajectories subsets training.
    train_dataset_full = TrajectoryDPFCached(
        str(cfg.train_h5), trajectory_length=1000, qpos_representation_override=qpos_override
    )
    val_dataset = TrajectoryDPFCached(
        str(cfg.val_h5), trajectory_length=1000, qpos_representation_override=qpos_override
    )
    if cfg.max_trajectories and cfg.max_trajectories < len(train_dataset_full):
        train_dataset = torch.utils.data.Subset(train_dataset_full, range(cfg.max_trajectories))
    else:
        train_dataset = train_dataset_full

    train_loader = DataLoader(
        train_dataset,
        batch_size=cfg.batch_size,
        shuffle=True,
        num_workers=cfg.num_workers,
        pin_memory=torch.cuda.is_available(),
        persistent_workers=(cfg.num_workers > 0),
    )
    val_loader = DataLoader(
        val_dataset,
        batch_size=cfg.batch_size,
        shuffle=False,
        num_workers=cfg.num_workers,
        pin_memory=torch.cuda.is_available(),
        persistent_workers=(cfg.num_workers > 0),
    )
    stats_loader = DataLoader(
        train_dataset,
        batch_size=cfg.batch_size,
        shuffle=False,
        num_workers=cfg.num_workers,
    )
    return train_dataset_full, train_dataset, val_dataset, train_loader, val_loader, stats_loader


def build_reacher_dpf_model(cfg: ReacherDPFTrainConfig, train_dataset_full, stats_loader):
    # Build dimensions and normalization ranges directly from the H5 data.
    sample = train_dataset_full[0]
    qpos_dim = sample['seq_qpos'].shape[-1]
    mom_dim = sample['seq_mom'].shape[-1]
    torque_dim = sample['seq_torque'].shape[-1]
    max_timesteps = train_dataset_full.num_steps
    qpos_representation = getattr(train_dataset_full, 'qpos_representation', 'raw')

    qpos_min, qpos_max, mom_min, mom_max, torque_min, torque_max = compute_normalization_stats(
        stats_loader,
        qpos_dim,
        mom_dim,
        torque_dim,
        max_timesteps,
        qpos_representation=qpos_representation,
    )

    return TrajectoryDPF(
        qpos_dim=qpos_dim,
        mom_dim=mom_dim,
        torque_dim=torque_dim,
        max_timesteps=max_timesteps,
        diffusion_steps=cfg.diffusion_steps,
        num_latents=cfg.num_latents,
        num_latent_channels=cfg.num_latent_channels,
        cond_dim=256,
        num_decoder_blocks=cfg.num_decoder_blocks,
        trajectory_length_training_options=(100, 200, 300, 400, 500, 600, 700, 800, 900, 1000),
        lr=cfg.lr,
        weight_decay=cfg.weight_decay,
        adam_beta1=cfg.adam_beta1,
        adam_beta2=cfg.adam_beta2,
        warmup_steps=cfg.warmup_steps,
        grad_warn_threshold=10.0,
        use_fused_adamw=False,
        encoder_cond_mode='none',
        backbone=cfg.backbone,
        conditioning_mode=cfg.conditioning_mode,
        query_context_mode=cfg.query_context_mode,
        token_layout=cfg.token_layout,
        dt=train_dataset_full.dt,
        data_dt=train_dataset_full.data_dt,
        xml_content=train_dataset_full.xml,
        qpos_representation=qpos_representation,
        qpos_min=qpos_min,
        qpos_max=qpos_max,
        mom_min=mom_min,
        mom_max=mom_max,
        torque_min=torque_min,
        torque_max=torque_max,
    )


def train_reacher_dpf(cfg: ReacherDPFTrainConfig = TRAIN_CFG):
    cfg.checkpoint_dir.mkdir(parents=True, exist_ok=True)
    if torch.cuda.is_available():
        torch.set_float32_matmul_precision('high')

    train_dataset_full, train_dataset, val_dataset, train_loader, val_loader, stats_loader = make_reacher_dpf_dataloaders(cfg)
    model = build_reacher_dpf_model(cfg, train_dataset_full, stats_loader)

    length_tag = 'VariableTrajLength'
    checkpoint_callback = ModelCheckpoint(
        dirpath=str(cfg.checkpoint_dir),
        filename=(
            'trajectory_dpf_x0Stabilized&AbsoluteTimeEncoding&'
            f'{length_tag}&UniformContext&EncoderNone&DecoderAttentions'
            '_cond-concat_torque_in_state_layout-shifted_tau_qpos-raw:{epoch:03d}_val_loss:{val_loss:.4f}'
        ),
        every_n_epochs=cfg.checkpoint_every_n_epochs,
    )

    trainer = pl.Trainer(
        max_epochs=cfg.epochs,
        accelerator='gpu' if torch.cuda.is_available() else 'cpu',
        devices=list(cfg.devices) if torch.cuda.is_available() else 1,
        callbacks=[checkpoint_callback],
        logger=False if cfg.disable_wandb else None,
        gradient_clip_val=cfg.grad_clip_val,
        gradient_clip_algorithm='norm',
        check_val_every_n_epoch=cfg.check_val_every_n_epoch,
        num_sanity_val_steps=cfg.num_sanity_val_steps,
        log_every_n_steps=10,
    )
    trainer.fit(model, train_loader, val_loader)
    return model, trainer


# Uncomment to train from scratch. For a quick smoke run first, set TRAIN_CFG.max_trajectories=128 and TRAIN_CFG.epochs=1.
# trained_model, trainer = train_reacher_dpf(TRAIN_CFG)


## 6. Select and Load a DPF Checkpoint

If you train from this notebook, `ACTIVE_CKPT_PATH` will automatically pick the newest checkpoint in your user-local checkpoint directory. If no local checkpoint exists, it falls back to the shared pretrained checkpoint on `/Data`. You can also set `MANUAL_CKPT_PATH` explicitly.


In [ ]:
import torch

# Option A: leave as None to use the newest checkpoint from CKPT_DIR.
# Option B: set this to an existing checkpoint path, for example a shared pre-trained checkpoint.
MANUAL_CKPT_PATH = None

def newest_checkpoint(checkpoint_dir: Path):
    ckpts = sorted(checkpoint_dir.glob('*.ckpt'), key=lambda p: p.stat().st_mtime)
    return ckpts[-1] if ckpts else None

# Load a user-trained checkpoint first; fall back to the shared pretrained checkpoint.
ACTIVE_CKPT_PATH = Path(MANUAL_CKPT_PATH) if MANUAL_CKPT_PATH else newest_checkpoint(CKPT_DIR)
if ACTIVE_CKPT_PATH is None and SHARED_PRETRAINED_CKPT.exists():
    ACTIVE_CKPT_PATH = SHARED_PRETRAINED_CKPT
if ACTIVE_CKPT_PATH is None:
    raise FileNotFoundError(
        f'No checkpoint found in {CKPT_DIR}, and shared checkpoint is missing. '
        'Run the training cell first, or set MANUAL_CKPT_PATH to an existing .ckpt file.'
    )

device = DEVICE if torch.cuda.is_available() or DEVICE == 'cpu' else 'cpu'
model = TrajectoryDPF.load_from_checkpoint(ACTIVE_CKPT_PATH, map_location=device)
model = model.to(device).eval()

print('checkpoint:', ACTIVE_CKPT_PATH)
print('device:', device)
print('qpos_dim:', model.qpos_dim)
print('mom_dim:', model.mom_dim)
print('torque_dim:', model.torque_dim)
print('state_dim:', model.state_dim)
print('conditioning_mode:', model.conditioning_mode)
print('token_layout:', model.token_layout)


## 7. Select and Load an HNN Checkpoint

This optional checkpoint is used for HNN-guided DPF sampling. If you only want unguided DPF, you can skip this cell.


In [ ]:

# Option A: leave as None to use the newest local HNN checkpoint, then the shared pretrained HNN.
# Option B: set this to an existing HNN .ckpt path.
MANUAL_HNN_CKPT_PATH = None

# Same lookup rule for HNN: local retrained checkpoint first, shared checkpoint second.
ACTIVE_HNN_CKPT_PATH = Path(MANUAL_HNN_CKPT_PATH) if MANUAL_HNN_CKPT_PATH else newest_checkpoint(HNN_CKPT_DIR)
if ACTIVE_HNN_CKPT_PATH is None and SHARED_HNN_CKPT.exists():
    ACTIVE_HNN_CKPT_PATH = SHARED_HNN_CKPT
if ACTIVE_HNN_CKPT_PATH is None:
    hnn_model = None
    print('No HNN checkpoint found; HNN-guided sampling will be unavailable until train_reacher_hnn() is run.')
else:
    hnn_model = HNNWrapper.load_from_checkpoint(ACTIVE_HNN_CKPT_PATH, map_location=device)
    hnn_model = hnn_model.to(device).eval()
    print('HNN checkpoint:', ACTIVE_HNN_CKPT_PATH)
    print('HNN q_std:', hnn_model.q_std.detach().cpu().numpy())
    print('HNN torque_gain:', hnn_model.torque_gain.detach().cpu().numpy())


## 8. Minimal Prefix-Completion Sampling Example

This cell samples one short completion from a validation trajectory prefix. The sampler fixes only the clean prefix, generates the noisy suffix, and uses a random subset of the full query as Perceiver context.


In [ ]:
import numpy as np

trajectory_length = 128
prefix_len = 16
num_samples = 1

with h5py.File(VAL_H5, 'r') as f:
    traj = f['traj_0']
    qpos = torch.tensor(traj['seq_qpos'][:trajectory_length], dtype=torch.float32, device=device).unsqueeze(0)
    mom = torch.tensor(traj['seq_mom'][:trajectory_length], dtype=torch.float32, device=device).unsqueeze(0)
    tau = torch.tensor(traj['seq_torque'][:trajectory_length], dtype=torch.float32, device=device).unsqueeze(0)


generated_state, generated_tau = model.sample_trajectories(
    num_samples=num_samples,
    trajectory_length=trajectory_length,
    num_diffusion_steps=20,
    prefix_len=prefix_len,
    observed_qpos=qpos,
    observed_mom=mom,
    observed_torque=tau,
    use_ema=False,
    sampler='ddim',
    )

print('generated_state:', tuple(generated_state.shape))
print('generated_tau:', tuple(generated_tau.shape))
print('first generated qpos:', generated_state[0, 0, :model.qpos_dim].detach().cpu().numpy())


## 9. Guidance Presets for Inference

Choose the guidance mode used by the candidate inference cell below. The default is unguided DPF; switch `ACTIVE_GUIDANCE_PRESET` to `hnn`, `target_l1`, `target_l2`, `target_linf`, `hnn_target_*`, or `target_hnn_*` when needed. `hnn_target_*` applies HNN guidance first and target guidance second; `target_hnn_*` applies target guidance first and HNN guidance second.


In [ ]:

# Base HNN guidance scale. Increase cautiously; too large can distort DPF samples.
HNN_GUIDANCE = {
    'hnn_model': globals().get('hnn_model', None),
    'alpha_q': 1e-4,
    'alpha_p': 1e-4,
}

# Target guidance scale/norm options for pulling the suffix end-effector toward target_xy.
TARGET_GUIDANCE = {
    'l1': {'target_guidance_alpha': 1e-3, 'target_guidance_norm': 'l1'},
    'l2': {'target_guidance_alpha': 1e-3, 'target_guidance_norm': 'l2'},
    'linf': {'target_guidance_alpha': 1e-3, 'target_guidance_norm': 'linf'},
}

GUIDANCE_PRESETS = {
    'unguided': {},
    'hnn': dict(HNN_GUIDANCE),
}

# Generate single-guidance and combined-guidance presets for each target norm.
for norm_name, target_kwargs in TARGET_GUIDANCE.items():
    GUIDANCE_PRESETS[f'target_{norm_name}'] = dict(target_kwargs)
    GUIDANCE_PRESETS[f'hnn_target_{norm_name}'] = {
        **HNN_GUIDANCE,
        **target_kwargs,
        'guidance_order': 'hnn_then_target',
    }
    GUIDANCE_PRESETS[f'target_hnn_{norm_name}'] = {
        **target_kwargs,
        **HNN_GUIDANCE,
        'guidance_order': 'target_then_hnn',
    }

# Change this string before running the candidate-inference cell.
ACTIVE_GUIDANCE_PRESET = 'unguided'
ACTIVE_GUIDANCE_KWARGS = dict(GUIDANCE_PRESETS[ACTIVE_GUIDANCE_PRESET])

if ACTIVE_GUIDANCE_KWARGS.get('hnn_model', None) is None:
    # Keep combined/HNN presets safe if the HNN checkpoint cell was skipped or unavailable.
    ACTIVE_GUIDANCE_KWARGS.pop('hnn_model', None)
    if 'hnn' in ACTIVE_GUIDANCE_PRESET:
        print('Warning: HNN preset selected but hnn_model is None; running without HNN guidance.')

print('available guidance presets:', ', '.join(GUIDANCE_PRESETS.keys()))
print('active guidance preset:', ACTIVE_GUIDANCE_PRESET)
print('active guidance kwargs:', {k: ('HNNWrapper' if k == 'hnn_model' else v) for k, v in ACTIVE_GUIDANCE_KWARGS.items()})


## 10. Reacher DPF Candidate Inference

This cell samples DPF suffix candidates from a source prefix and scores them by closest predicted end-effector distance to a target point. The guidance behavior is controlled by `ACTIVE_GUIDANCE_PRESET` and `ACTIVE_GUIDANCE_KWARGS` from the previous section.


In [ ]:
import matplotlib.pyplot as plt


def reacher_fingertip_xy(qpos):
    """Forward kinematics for the 2-link Reacher fingertip used by this dataset."""
    q = np.asarray(qpos)
    q0 = q[..., 0]
    q1 = q[..., 1]
    x = 0.1 * np.cos(q0) + 0.11 * np.cos(q0 + q1)
    y = 0.1 * np.sin(q0) + 0.11 * np.sin(q0 + q1)
    return np.stack([x, y], axis=-1)


def sample_dpf_candidates_from_prefix(
    model,
    h5_path: Path,
    source_traj: str = 'traj_0',
    target_traj: str = 'traj_1',
    source_start: int = 100,
    target_index: int = 700,
    prefix_len: int = 16,
    lookahead_steps: int = 256,
    num_candidates: int = 128,
    num_diffusion_steps: int = 20,
    hnn_model=None,
    alpha_q: float = 0.0,
    alpha_p: float = 0.0,
    target_guidance_alpha: float = 0.0,
    target_guidance_norm: str = 'l2',
    guidance_order: str = 'hnn_then_target',
):
    # The candidate trajectory is the observed prefix plus a generated suffix.
    horizon = prefix_len + lookahead_steps
    with h5py.File(h5_path, 'r') as f:
        src = f[source_traj]
        tgt = f[target_traj]
        src_slice = slice(source_start, source_start + prefix_len)
        qpos_np = src['seq_qpos'][src_slice].astype('float32')
        mom_np = src['seq_mom'][src_slice].astype('float32')
        tau_np = src['seq_torque'][src_slice].astype('float32')
        # The target trajectory only provides a goal qpos; scoring/guidance use its fingertip xy.
        target_qpos = tgt['seq_qpos'][target_index].astype('float32')
        target_xy = reacher_fingertip_xy(target_qpos)

    # Fill only the prefix with real states. The suffix entries are placeholders for the sampler.
    observed_qpos = torch.zeros((1, horizon, model.qpos_dim), dtype=torch.float32, device=device)
    observed_mom = torch.zeros((1, horizon, model.mom_dim), dtype=torch.float32, device=device)
    observed_tau = torch.zeros((1, horizon, model.torque_dim), dtype=torch.float32, device=device)
    observed_qpos[0, :prefix_len] = torch.from_numpy(qpos_np).to(device)
    observed_mom[0, :prefix_len] = torch.from_numpy(mom_np).to(device)
    observed_tau[0, :prefix_len] = torch.from_numpy(tau_np).to(device)

    # sample_trajectories expands this one prefix into num_candidates independent completions.
    generated_state, generated_tau = model.sample_trajectories(
        num_samples=num_candidates,
        trajectory_length=horizon,
        num_diffusion_steps=num_diffusion_steps,
        prefix_len=prefix_len,
        observed_qpos=observed_qpos,
        observed_mom=observed_mom,
        observed_torque=observed_tau,
        use_ema=False,
        sampler='ddim',
        hnn=hnn_model,
        alpha_q=alpha_q,
        alpha_p=alpha_p,
        target_xy=torch.as_tensor(target_xy, dtype=torch.float32, device=device),
        target_guidance_alpha=target_guidance_alpha,
        target_guidance_norm=target_guidance_norm,
        guidance_order=guidance_order,
    )

    state_np = generated_state.detach().cpu().numpy()
    tau_out_np = generated_tau.detach().cpu().numpy()
    qpos_candidates = state_np[:, :, : model.qpos_dim]
    ee_candidates = reacher_fingertip_xy(qpos_candidates)
    # Pick the candidate whose generated suffix gets closest to the target at any future step.
    suffix_ee = ee_candidates[:, prefix_len:]
    dists = np.linalg.norm(suffix_ee - target_xy.reshape(1, 1, 2), axis=-1)
    best_per_candidate = dists.min(axis=1)
    best_idx = int(best_per_candidate.argmin())

    return {
        'source_traj': source_traj,
        'target_traj': target_traj,
        'source_start': int(source_start),
        'target_index': int(target_index),
        'prefix_len': int(prefix_len),
        'lookahead_steps': int(lookahead_steps),
        'num_candidates': int(num_candidates),
        'num_diffusion_steps': int(num_diffusion_steps),
        'hnn_guided': hnn_model is not None,
        'alpha_q': float(alpha_q),
        'alpha_p': float(alpha_p),
        'target_guidance_alpha': float(target_guidance_alpha),
        'target_guidance_norm': str(target_guidance_norm),
        'guidance_order': str(guidance_order),
        'target_xy': target_xy.tolist(),
        'best_candidate_index': best_idx,
        'best_goal_distance': float(best_per_candidate[best_idx]),
        'qpos_candidates': qpos_candidates,
        'ee_candidates': ee_candidates,
        'tau_candidates': tau_out_np,
    }


def plot_best_candidate(result, output_dir: Path = EVAL_OUTPUT_DIR):
    output_dir.mkdir(parents=True, exist_ok=True)
    best_idx = result['best_candidate_index']
    ee = result['ee_candidates'][best_idx]
    prefix_len = result['prefix_len']
    target_xy = np.asarray(result['target_xy'])

    # Plot the best predicted end-effector path in the same source-prefix/target style used in reports.
    fig, ax = plt.subplots(figsize=(5, 5), dpi=160)
    ax.plot(ee[:prefix_len, 0], ee[:prefix_len, 1], color='black', linewidth=2, label='observed prefix')
    ax.plot(ee[prefix_len:, 0], ee[prefix_len:, 1], color='tab:blue', alpha=0.85, label='generated suffix')
    ax.scatter(ee[0, 0], ee[0, 1], color='tab:green', s=40, label='start')
    ax.scatter(target_xy[0], target_xy[1], color='tab:red', marker='*', s=120, label='target')
    ax.set_aspect('equal', adjustable='box')
    ax.set_xlabel('x')
    ax.set_ylabel('y')
    ax.set_title(f"Best candidate, min goal dist={result['best_goal_distance']:.4f}")
    ax.legend(loc='best')
    fig.tight_layout()
    fig_path = output_dir / 'notebook_dpf_best_candidate.png'
    fig.savefig(fig_path)
    plt.show()
    return fig_path


candidate_result = sample_dpf_candidates_from_prefix(
    model,
    VAL_H5,
    num_candidates=32,
    **ACTIVE_GUIDANCE_KWARGS,
)
figure_path = plot_best_candidate(candidate_result)
print('best candidate index:', candidate_result['best_candidate_index'])
print('best predicted goal distance:', candidate_result['best_goal_distance'])
print('figure saved to:', figure_path)



The candidate-inference cell above is intentionally modest (`num_candidates=32`) so it can be run interactively. Increase to `128` to match the main unguided DPF candidate setting.


In [ ]:
# Full-size candidate inference example:
# candidate_result = sample_dpf_candidates_from_prefix(model, VAL_H5, num_candidates=128, **ACTIVE_GUIDANCE_KWARGS)
# figure_path = plot_best_candidate(candidate_result)


## 11. Expected Outputs

The inference code writes the following plot under `EVAL_OUTPUT_DIR`:

- `notebook_dpf_best_candidate.png`

DPF checkpoints are written under user-local `CKPT_DIR`. HNN checkpoints are written under user-local `HNN_CKPT_DIR`. The shared dataset plus pretrained DPF/HNN checkpoints live under `/Data`, so another user on this server can run the notebook without accessing another user's home directory or importing this repository's Python modules.
